# ESPN API — Real-Time Game Data Test Cases

This notebook demonstrates the best ESPN endpoints for retrieving **live, in-progress game data**.
We test against real ongoing games and compare response structures across API versions.

## Endpoint Tiers for Live Data

| Tier | Endpoint | Best For |
|------|----------|----------|
| **1 (Best)** | `site.api.espn.com/apis/site/v2/sports/{sport}/{league}/summary?event={id}` | Everything in one call: live scores, play-by-play, win probability, current situation, odds, boxscore |
| **2 (Great)** | `site.api.espn.com/apis/site/v2/sports/{sport}/{league}/scoreboard` | Discovering which games are live right now |
| **3 (Granular)** | `sports.core.api.espn.com/v2/...` | Individual resources (situation, odds, plays, probabilities) via `$ref` URLs |
| **4 (CDN)** | `cdn.espn.com/core/{sport}/game?xhr=1&gameId={id}` | Lightweight, CDN-cached (may be stale) |

## Live Games Available Right Now
- **MLB**: Multiple games in progress
- **NFL**: Broncos at Chiefs (in progress)


In [13]:
import httpx
import json
import pprint

def show(data, max_chars=800):
    """Pretty-print JSON, truncated for readability."""
    text = json.dumps(data, indent=2)
    if len(text) > max_chars:
        text = text[:max_chars] + f"\n... (truncated, {len(json.dumps(data, indent=2))} total chars)"
    print(text)


---
## Test 1: Scoreboard — Discover Live Games

**Endpoint:** `GET site.api.espn.com/apis/site/v2/sports/{sport}/{league}/scoreboard`

**Why use it:** This is your entry point. It tells you which games are live *right now* and gives you their event IDs.
You need those IDs before you can query any game-specific endpoint.

**Key fields per event:**
- `id` — the event ID you need for summary/situation/odds calls
- `status.type.state` — `"pre"` | `"in"` | `"post"`
- `status.type.description` — human-readable status (e.g. "In Progress", "Final")
- `competitions[0].competitors[]` — teams with current scores


In [20]:
# --- MLB Scoreboard: Find live games ---
SPORT = "baseball"
LEAGUE = "mlb"
url = f"https://site.api.espn.com/apis/site/v2/sports/{SPORT}/{LEAGUE}/scoreboard"

r = httpx.get(url, timeout=10)
r.raise_for_status()
data = r.json()

print(f"Total events: {len(data.get('events', []))}")
print()

live_games = []
for e in data.get("events", []):
    status = e["status"]["type"]
    state = status["state"]
    desc = status["description"]
    
    # Get scores
    comp = e.get("competitions", [{}])[0]
    scores = {}
    for team in comp.get("competitors", []):
        name = team["team"]["abbreviation"]
        scores[team["homeAway"]] = {"team": name, "score": team.get("score", "N/A")}
    
    marker = "🔴 LIVE" if state == "in" else ("⏳" if state == "pre" else "✅")
    print(f"{marker} ID: {e['id']} | {e['name']}")
    print(f"   Status: {desc} | Date: {e['date']}")
    if scores:
        away = scores.get("away", {})
        home = scores.get("home", {})
        print(f"   Score: {away.get('team', '?')} {away.get('score', '?')} - {home.get('score', '?')} {home.get('team', '?')}")
    print()
    
    if state == "in":
        live_games.append(e["id"])

print(f"\nLive game IDs: {live_games}")


Total events: 10

✅ ID: 401816934 | Chicago White Sox at Cleveland Guardians
   Status: Final | Date: 2026-09-14T22:40Z
   Score: CHW 7 - 3 CLE

✅ ID: 401816936 | Los Angeles Dodgers at Cincinnati Reds
   Status: Final | Date: 2026-09-14T22:40Z
   Score: LAD 4 - 1 CIN

✅ ID: 401816935 | Detroit Tigers at Toronto Blue Jays
   Status: Final | Date: 2026-09-14T23:07Z
   Score: DET 6 - 5 TOR

✅ ID: 401816933 | Baltimore Orioles at New York Mets
   Status: Final | Date: 2026-09-14T23:10Z
   Score: BAL 2 - 1 NYM

✅ ID: 401816937 | Atlanta Braves at Chicago Cubs
   Status: Final | Date: 2026-09-14T23:40Z
   Score: ATL 3 - 7 CHC

✅ ID: 401816938 | New York Yankees at Minnesota Twins
   Status: Final | Date: 2026-09-14T23:40Z
   Score: NYY 8 - 3 MIN

✅ ID: 401816939 | San Francisco Giants at St. Louis Cardinals
   Status: Final | Date: 2026-09-14T23:45Z
   Score: SF 1 - 2 STL

✅ ID: 401816940 | San Diego Padres at Colorado Rockies
   Status: Final | Date: 2026-09-15T00:40Z
   Score: SD 8 - 7 CO

---
## Test 2: Game Summary — The "Everything" Endpoint

**Endpoint:** `GET site.api.espn.com/apis/site/v2/sports/{sport}/{league}/summary?event={event_id}`

**Why use it:** This is the single best endpoint for live game data. It returns a massive payload with:
- `situation` — current game state (inning, down/distance, possession, etc.)
- `plays` — full play-by-play log
- `winprobability` — ESPN's live win probability model
- `pickcenter` / `odds` — betting odds from multiple sportsbooks
- `boxscore` — player and team stats
- `header` — score summary
- `drives` — drive summary (NFL only)
- `againstTheSpread` — ATS records

**This is the endpoint you should poll during live games.**


In [16]:
# --- Pick a live game and fetch its summary ---
if not live_games:
    print("No live games right now. Using a known recent game ID instead.")
    EVENT_ID = "401816934"  # White Sox at Guardians (was live 2026-09-14)
else:
    EVENT_ID = live_games[0]

SUMMARY_URL = f"https://site.api.espn.com/apis/site/v2/sports/{SPORT}/{LEAGUE}/summary?event={EVENT_ID}"
print(f"Fetching: {SUMMARY_URL}")

r = httpx.get(SUMMARY_URL, timeout=10)
r.raise_for_status()
summary = r.json()

# Show what's available
print("\n=== AVAILABLE SECTIONS ===")
for key in sorted(summary.keys()):
    val = summary[key]
    if isinstance(val, list):
        print(f"  {key}: list[{len(val)}]")
    elif isinstance(val, dict):
        print(f"  {key}: dict[{len(val)} keys]")
    else:
        print(f"  {key}: {type(val).__name__}")


Fetching: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/summary?event=401816934

=== AVAILABLE SECTIONS ===
  againstTheSpread: list[2]
  atBats: dict[58 keys]
  boxscore: dict[2 keys]
  broadcasts: list[3]
  format: dict[1 keys]
  gameInfo: dict[3 keys]
  header: dict[7 keys]
  injuries: list[2]
  meta: dict[8 keys]
  news: dict[3 keys]
  notes: list[0]
  odds: list[0]
  pickcenter: list[1]
  plays: list[469]
  playsMap: dict[469 keys]
  rosters: list[2]
  seasonseries: list[2]
  situation: dict[7 keys]
  standings: dict[2 keys]
  videos: list[0]
  wallclockAvailable: bool
  winprobability: list[57]


---
## Test 3: Live Situation — What's Happening Right Now

**From summary:** `summary["situation"]`

**Also available as standalone:** `GET sports.core.api.espn.com/v2/sports/{sport}/leagues/{league}/events/{id}/competitions/{id}/situation`

This tells you the exact moment-by-moment state of the game. For MLB: balls, strikes, outs, who's on base,
current pitcher/batter. For NFL: down, distance, yard line, possession.


In [17]:
# --- Live Situation ---
situation = summary.get("situation", {})
if situation:
    print("=== CURRENT SITUATION ===")
    show(situation, max_chars=2000)
else:
    print("No situation data (game may not be in progress)")


=== CURRENT SITUATION ===
{
  "lastPlay": {
    "id": "4018169341301010001"
  },
  "balls": 0,
  "strikes": 0,
  "outs": 0,
  "pitcher": {
    "playerId": 4867679
  },
  "batter": {
    "playerId": 5007707
  },
  "situationNotes": [
    {
      "type": "BVP_STATS",
      "text": "Bazzana career vs. Burke: 0 for 6 (.000 AVG), 0 HR, 4 SO"
    },
    {
      "type": "CHANCES_TO_SCORE_1PLUS",
      "text": "Chance of scoring 1+ runs this inning (0 outs, Bases Empty): 29.41%"
    },
    {
      "type": "CHANCES_TO_SCORE_2PLUS",
      "text": "Chance of scoring 2+ runs this inning (0 outs, Bases Empty): 14.10%"
    }
  ]
}


---
## Test 4: Live Win Probability

**From summary:** `summary["winprobability"]`

**Also available as standalone:** `GET sports.core.api.espn.com/v2/sports/{sport}/leagues/{league}/events/{id}/competitions/{id}/probabilities`

ESPN's model calculates win probability after every play. Each entry links to a `playId` so you can
correlate probability shifts with specific plays. This is extremely valuable for odds engines.


In [18]:
# --- Win Probability ---
wp = summary.get("winprobability", [])
print(f"Total win probability entries: {len(wp)}")
print()

if wp:
    print("=== LAST 5 WIN PROBABILITY SHIFTS ===")
    for entry in wp[-5:]:
        home_pct = entry.get("homeWinPercentage", 0) * 100
        away_pct = (1 - entry.get("homeWinPercentage", 0)) * 100
        tie_pct = entry.get("tiePercentage", 0) * 100
        play_id = entry.get("playId", "N/A")
        print(f"  Play {play_id}: Home {home_pct:.1f}% | Away {away_pct:.1f}% | Tie {tie_pct:.1f}%")
    
    # Plot if matplotlib available
    try:
        import matplotlib.pyplot as plt
        home_pcts = [e.get("homeWinPercentage", 0.5) * 100 for e in wp]
        plt.figure(figsize=(10, 4))
        plt.plot(home_pcts, linewidth=2, color="#1a73e8")
        plt.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
        plt.ylabel("Home Win %")
        plt.xlabel("Play #")
        plt.title(f"Live Win Probability — Event {EVENT_ID}")
        plt.ylim(0, 100)
        plt.show()
    except ImportError:
        print("\n(matplotlib not available for plotting)")


Total win probability entries: 57

=== LAST 5 WIN PROBABILITY SHIFTS ===
  Play 4018169341105990057: Home 5.1% | Away 94.9% | Tie 0.0%
  Play 4018169341106990057: Home 5.7% | Away 94.3% | Tie 0.0%
  Play 4018169341201990057: Home 3.1% | Away 96.9% | Tie 0.0%
  Play 4018169341202990057: Home 3.3% | Away 96.7% | Tie 0.0%
  Play 4018169341203990057: Home 3.4% | Away 96.6% | Tie 0.0%

(matplotlib not available for plotting)


---
## Test 5: Betting Odds

**From summary:** `summary["pickcenter"]` (live odds) and `summary["odds"]`

**Also available as standalone:** `GET sports.core.api.espn.com/v2/sports/{sport}/leagues/{league}/events/{id}/competitions/{id}/odds`

The `pickcenter` array contains odds from multiple sportsbooks. Each entry includes:
- `provider` — sportsbook name (DraftKings, FanDuel, etc.)
- `details` — human-readable odds summary
- `spread`, `overUnder`, `overOdds`, `underOdds`
- `awayTeamOdds` / `homeTeamOdds` — moneyline, spread, total with open/close values


In [19]:
# --- Betting Odds ---
pickcenter = summary.get("pickcenter", [])
odds = summary.get("odds", [])

print(f"Pickcenter entries: {len(pickcenter)}")
print(f"Odds entries: {len(odds)}")
print()

if pickcenter:
    print("=== LIVE ODDS (Pickcenter) ===")
    for pc in pickcenter:
        provider = pc.get("provider", {}).get("name", "Unknown")
        details = pc.get("details", "N/A")
        spread = pc.get("spread", "N/A")
        ou = pc.get("overUnder", "N/A")
        over_odds = pc.get("overOdds", "N/A")
        under_odds = pc.get("underOdds", "N/A")
        
        print(f"\n  📖 {provider}")
        print(f"     Line: {details}")
        print(f"     Spread: {spread} | O/U: {ou}")
        print(f"     Over: {over_odds} | Under: {under_odds}")
        
        # Team-level odds
        for side in ["awayTeamOdds", "homeTeamOdds"]:
            team_odds = pc.get(side, {})
            if team_odds:
                team_name = side.replace("TeamOdds", "").upper()
                ml = team_odds.get("moneyLine", "N/A")
                fav = team_odds.get("favorite", False)
                print(f"     {team_name} MoneyLine: {ml} {'(Favorite)' if fav else '(Underdog)'}")


Pickcenter entries: 1
Odds entries: 0

=== LIVE ODDS (Pickcenter) ===

  📖 DraftKings
     Line: CLE -184
     Spread: -1.5 | O/U: 7.0
     Over: -104.0 | Under: -116.0
     AWAY MoneyLine: 152 (Underdog)
     HOME MoneyLine: -184 (Favorite)


---
## Test 6: Play-by-Play

**From summary:** `summary["plays"]`

**Also available as standalone:** `GET sports.core.api.espn.com/v2/sports/{sport}/leagues/{league}/events/{id}/competitions/{id}/plays`

Every play is logged with type, text description, score at that moment, period, and participants.
The `scoringPlay` flag lets you filter for scoring plays only.


In [ ]:
# --- Play-by-Play ---
plays = summary.get("plays", [])
print(f"Total plays logged: {len(plays)}")

if plays:
    # Show last 5 plays
    print("\n=== LAST 5 PLAYS ===")
    for p in plays[-5:]:
        period = p.get("period", {}).get("displayValue", "?")
        ptype = p.get("type", {}).get("text", "?")
        text = p.get("text", "?")
        away_score = p.get("awayScore", "?")
        home_score = p.get("homeScore", "?")
        scoring = " 🏈 SCORE!" if p.get("scoringPlay") else ""
        print(f"  [{period}] ({away_score}-{home_score}) {ptype}: {text}{scoring}")
    
    # Show only scoring plays
    scoring_plays = [p for p in plays if p.get("scoringPlay")]
    print(f"\n=== SCORING PLAYS ({len(scoring_plays)} total) ===")
    for p in scoring_plays:
        period = p.get("period", {}).get("displayValue", "?")
        text = p.get("text", "?")
        away_score = p.get("awayScore", "?")
        home_score = p.get("homeScore", "?")
        print(f"  [{period}] ({away_score}-{home_score}) {text}")


---
## Test 7: Boxscore — Player & Team Stats

**From summary:** `summary["boxscore"]`

Contains team stats, player stats, and scoring summaries. Structure varies by sport.


In [ ]:
# --- Boxscore ---
boxscore = summary.get("boxscore", {})
if boxscore:
    print("=== BOXSCORE SECTIONS ===")
    for key in boxscore:
        val = boxscore[key]
        if isinstance(val, list):
            print(f"  {key}: list[{len(val)}]")
        elif isinstance(val, dict):
            print(f"  {key}: dict[{len(val)} keys]")
        else:
            print(f"  {key}: {type(val).__name__}")
    
    # Show team stats if available
    teams = boxscore.get("teams", [])
    if teams:
        print("\n=== TEAM STATS ===")
        for team_entry in teams:
            team_name = team_entry.get("team", {}).get("displayName", "Unknown")
            print(f"\n  {team_name}:")
            for stat in team_entry.get("statistics", [])[:5]:
                label = stat.get("label", stat.get("name", "?"))
                value = stat.get("displayValue", "?")
                print(f"    {label}: {value}")
else:
    print("No boxscore available")


---
## Test 8: NFL Drives (Football Only)

**From summary:** `summary["drives"]`

For NFL games, the summary includes drive-by-drive data with field position, play count,
yards gained, time of possession, and result.


In [ ]:
# --- NFL Drives (switch to NFL if there's a live game) ---
NFL_EVENT_ID = "401872931"  # Broncos at Chiefs (was live 2026-09-14)

nfl_url = f"https://site.api.espn.com/apis/site/v2/sports/football/nfl/summary?event={NFL_EVENT_ID}"
r = httpx.get(nfl_url, timeout=10)
r.raise_for_status()
nfl_summary = r.json()

drives = nfl_summary.get("drives", {})
if drives:
    current = drives.get("current", {})
    print("=== CURRENT DRIVE ===")
    print(f"  {current.get('description', 'N/A')}")
    print(f"  Team: {current.get('team', {}).get('displayName', 'N/A')}")
    
    # Recent drives
    previous = drives.get("previous", [])
    if previous:
        print(f"\n=== PREVIOUS DRIVES (last 5 of {len(previous)}) ===")
        for d in previous[-5:]:
            desc = d.get("description", "N/A")
            team = d.get("team", {}).get("abbreviation", "?")
            result = d.get("result", "N/A")
            print(f"  [{team}] {desc} → {result}")
else:
    print("No drive data (not an NFL game or game not started)")


---
## Test 9: Core API v2 — Granular Resources

**Base:** `sports.core.api.espn.com/v2/sports/{sport}/leagues/{league}/events/{id}/competitions/{id}/...`

These are the "building block" endpoints. Each returns a focused slice of data.
Useful when you only need one specific thing (lighter than the full summary).

| Resource | What it returns |
|----------|----------------|
| `/situation` | Current game state (balls/strikes/outs, down/distance) |
| `/odds` | Betting odds with open/close history |
| `/probabilities` | Win probability entries |
| `/plays` | Play-by-play (paginated, use `?limit=` and `&page=`) |
| `/competitors/{id}/linescores` | Period-by-period scores |

**Note:** Many responses contain `$ref` URLs pointing to `sports.core.api.espn.pvt` — replace `.pvt` with `.com` to make them work.


In [ ]:
# --- Core API v2: Situation (standalone) ---
CORE_BASE = f"https://sports.core.api.espn.com/v2/sports/{SPORT}/leagues/{LEAGUE}"
situation_url = f"{CORE_BASE}/events/{EVENT_ID}/competitions/{EVENT_ID}/situation"

r = httpx.get(situation_url, timeout=10)
r.raise_for_status()
core_situation = r.json()

print("=== CORE API V2: SITUATION ===")
show(core_situation, max_chars=1500)


In [ ]:
# --- Core API v2: Odds (standalone) ---
odds_url = f"{CORE_BASE}/events/{EVENT_ID}/competitions/{EVENT_ID}/odds"

r = httpx.get(odds_url, timeout=10)
r.raise_for_status()
core_odds = r.json()

print("=== CORE API V2: ODDS ===")
print(f"Providers: {core_odds.get('count', 0)}")

for item in core_odds.get("items", []):
    provider = item.get("provider", {}).get("name", "Unknown")
    details = item.get("details", "N/A")
    spread = item.get("spread", "N/A")
    ou = item.get("overUnder", "N/A")
    print(f"\n  📖 {provider}")
    print(f"     {details} | Spread: {spread} | O/U: {ou}")
    
    # Show open vs close for away team
    away = item.get("awayTeamOdds", {})
    if away:
        open_odds = away.get("open", {})
        close_odds = away.get("close", {})
        if open_odds:
            ml_open = open_odds.get("moneyLine", {}).get("alternateDisplayValue", "N/A")
            print(f"     Away ML Open: {ml_open}")
        if close_odds:
            ml_close = close_odds.get("moneyLine", {}).get("alternateDisplayValue", "N/A")
            print(f"     Away ML Close: {ml_close}")


---
## Test 10: Polling Pattern — How to Poll During Live Games

For a live odds engine, you'll want to poll the summary endpoint at regular intervals during live games.
Here's a simple pattern with rate limiting:


In [ ]:
# --- Polling Pattern (runs for ~30 seconds as demo) ---
import time

def poll_live_game(sport, league, event_id, interval_seconds=10, max_polls=3):
    """Poll the summary endpoint for live game data."""
    url = f"https://site.api.espn.com/apis/site/v2/sports/{sport}/{league}/summary?event={event_id}"
    
    for i in range(max_polls):
        r = httpx.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        
        # Extract key live info
        header = data.get("header", {})
        competitions = header.get("competitions", [{}])
        status = header.get("status", {})
        
        state = status.get("type", {}).get("state", "unknown")
        desc = status.get("type", {}).get("description", "unknown")
        
        scores = {}
        if competitions:
            for team in competitions[0].get("competitors", []):
                abbr = team.get("team", {}).get("abbreviation", "?")
                scores[abbr] = team.get("score", "?")
        
        situation = data.get("situation", {})
        outs = situation.get("outs", "N/A")
        balls = situation.get("balls", "N/A")
        strikes = situation.get("strikes", "N/A")
        
        wp_data = data.get("winprobability", [])
        last_wp = wp_data[-1].get("homeWinPercentage", 0.5) * 100 if wp_data else 50.0
        
        print(f"Poll #{i+1} | {desc} | State: {state}")
        print(f"  Scores: {scores}")
        print(f"  Count: {balls}-{strikes}, Outs: {outs}")
        print(f"  Home Win Probability: {last_wp:.1f}%")
        print()
        
        if i < max_polls - 1:
            time.sleep(interval_seconds)

# Run for a live game (or any recent game)
poll_game_id = live_games[0] if live_games else EVENT_ID
print(f"Polling game {poll_game_id} every 10 seconds (3 polls)...\n")
poll_live_game(SPORT, LEAGUE, poll_game_id, interval_seconds=10, max_polls=3)


---
## Summary: Which Endpoints Should You Use?

### For Your Odds Engine — Recommended Approach:

| Use Case | Endpoint | Polling Frequency |
|----------|----------|-------------------|
| **Discover live games** | `site/v2/.../scoreboard` | Every 30-60s |
| **Get all live game data** | `site/v2/.../summary?event={id}` | Every 5-15s during live games |
| **Just current situation** | `core/v2/.../situation` | Every 3-5s (lightweight) |
| **Just odds** | `core/v2/.../odds` | Every 10-30s |
| **Just win probability** | `core/v2/.../probabilities` | Every 5-10s |

### Key Takeaways:
1. **The `summary` endpoint is your best friend** — one call gets you everything: scores, situation, plays, win probability, and odds
2. **Scoreboard first** — always check the scoreboard to find which games are live before hitting game-specific endpoints
3. **The Core API v2 is good for targeted polling** — if you only need odds, use `/odds` instead of the full summary (less data transfer)
4. **Win probability is play-linked** — each entry has a `playId`, so you can correlate probability shifts with specific plays
5. **Odds include open/close history** — the `awayTeamOdds.open` and `.close` objects show how lines moved
6. **Rate limiting:** Be respectful — poll summary every 5-15 seconds during live games, not every second
